# FinDirector — Directive Model Evaluation on Colab

Evaluate the fine-tuned Qwen 2.5 7B + LoRA directive model on the 156-example
held-out test set (greedy decoding, temperature 0).

**Setup:**
- Runtime: L4 GPU (24 GB VRAM)
- Estimated time: 5-20 minutes
- Cost: ~$0.15 in Colab compute units

**Prerequisites:**
- Colab Pro subscription with L4 GPU access
- Secrets configured: `HF_TOKEN` (adapter + dataset repos are private)

**Pipeline this notebook runs:**
1. Verify GPU allocation
2. Clone the FinDirector repo
3. Install requirements
4. Load `HF_TOKEN` from Colab Secrets manager
5. Run `scripts/evaluate_directive_model.py`
6. Read the accuracy / confusion-matrix report from the output

## Step 1 — Verify GPU

In [ ]:
!nvidia-smi

## Step 2 — Clone the FinDirector Repo

Get the latest evaluation script and config from GitHub.

In [ ]:
# Clone the repo (public HTTPS clone, no auth needed for public repos)
!git clone https://github.com/ShantanuBapat/findirector.git
%cd findirector
!git log -1 --oneline  # Show the latest commit for reference

## Step 3 — Install Requirements

Install all dependencies. Colab has some pre-installed but we force upgrade
to match our pinned versions (this also picks up scikit-learn for metrics).

In [ ]:
# Install all requirements
!pip install -q --upgrade pip
!pip install -q -r requirements.txt

## Step 4 — Load Secrets from Colab Secrets Manager

Access `HF_TOKEN` without hardcoding it. Needed because both the adapter
(`AlHindi/findirector-directive-lora`) and the dataset
(`AlHindi/findirector-splits`) are private repos.

In [ ]:
import os
from google.colab import userdata

# Load secret into environment variable
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

# Verify (without printing the actual value)
assert os.environ["HF_TOKEN"].startswith("hf_"), "HF_TOKEN not loaded correctly"

print("Secret loaded successfully")
print(f"HF_TOKEN length: {len(os.environ['HF_TOKEN'])}")

## Step 5 — Run Evaluation

Execute the evaluation script. It downloads the LoRA adapter + test split from
HF Hub, generates on all 156 examples, and prints the report to this output.

**Expected duration:** 5-20 minutes on L4 GPU.

In [ ]:
# Run evaluation; output (accuracy, per-code P/R/F1, confusion matrix) streams here
!python -m scripts.evaluate_directive_model

## Done!

The report above shows:
- **Overall accuracy** over all 156 test examples
- **Parse-error tally** (tracked separately from misclassifications)
- **Per-code** precision / recall / F1
- **Confusion matrix** (rows = true, cols = predicted)

**Next steps (Session 2.6 analysis):**
- Read per-code accuracy and the confusion matrix
- Inspect the compute-vs-lookup boundary (expected tricky pair)
- Document findings and commit the eval script's results